## Script to convert a VIA dataset (like Greenland OOD dataset) to a JSON file compatible with COCO format

In [ ]:
import pandas as pd
import json
import os
import cv2
import numpy as np
from tqdm import tqdm
import time

CSV_FILE = "detections.csv"
DATASETS_PATH = os.environ.get("ZERO_SHOT_DATA_DIR", "./data")
IMAGE_DIR = os.path.join(DATASETS_PATH, "rangex")
OUTPUT_JSON = os.path.join(DATASETS_PATH, "rangex", "detections_coco_last.json")

def parse_class_field(class_str):
    """Parse class field to extract category ID and name"""
    parts = class_str.strip().split(' ', 1)
    if len(parts) == 2 and parts[0].isdigit():
        return int(parts[0]), parts[1]
    return 0, class_str

def resolve_image_path(img_dir, filename):
    """
    Resolve the actual image path. 
    Annotations are always in the _2 version, so check for that first.
    """
    # Try _2 version
    name_without_ext = os.path.splitext(filename)[0]
    ext = os.path.splitext(filename)[1]
    path_2 = os.path.join(img_dir, f"{name_without_ext}_2{ext}")
    if os.path.exists(path_2):
        return path_2
    
    # Fall back to original filename
    path_original = os.path.join(img_dir, filename)
    if os.path.exists(path_original):
        return path_original
    
    # Try _1 version if neither exists
    path_1 = os.path.join(img_dir, f"{name_without_ext}_1{ext}")
    if os.path.exists(path_1):
        return path_1
    
    return None

def get_images(img_dir):
    image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp'}
    images = []
    
    for filename in os.listdir(img_dir):
        if os.path.isfile(os.path.join(img_dir, filename)):
            if os.path.splitext(filename)[1].lower() in image_extensions:
                images.append(filename)
    
    return sorted(images)

def create_coco(csv_path, img_dir):
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"Error: CSV file '{csv_path}' not found.")
        return

    categories_dict = {}
    for class_str in df['class'].unique():
        cat_id, cat_name = parse_class_field(str(class_str))
        if cat_id not in categories_dict:
            categories_dict[cat_id] = {
                "id": cat_id,
                "name": cat_name,
                "supercategory": "insect"
            }

    coco_output = {
        "info": {
            "description": "Converted from custom CSV format",
            "year": 2026,
            "date_created": time.strftime("%Y-%m-%d %H:%M:%S")
        },
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": sorted(categories_dict.values(), key=lambda x: x['id'])
    }

    filename_to_id = {}
    current_img_id = 1
    current_ann_id = 1
    skipped_rows = 0

    print(f"Processing {len(df)} rows from CSV...")
    print(f"Found {len(coco_output['categories'])} unique categories: {[c['name'] for c in coco_output['categories']]}")
    print(f"Image directory: {os.path.abspath(img_dir)}")

    all_images = get_images(img_dir)
    print(f"Found {len(all_images)} images in filesystem")
    
    for img_filename in tqdm(all_images, desc="Loading images"):
        img_path = os.path.join(img_dir, img_filename)
        
        try:
            img = cv2.imread(img_path)
            if img is not None:
                height, width = img.shape[:2]
            else:
                print(f"WARNING: cv2.imread returned None for {img_path}, using default dimensions")
                height, width = 0, 0
        except Exception as e:
            print(f"ERROR reading {img_path}: {e}, using default dimensions")
            height, width = 0, 0
        
        image_info = {
            "id": current_img_id,
            "file_name": img_filename,
            "width": width,
            "height": height
        }
        coco_output["images"].append(image_info)
        filename_to_id[img_filename] = current_img_id
        current_img_id += 1

    print(f"\nProcessing annotations from CSV...")
    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        filename = row['sourceFile']
        
        if filename not in filename_to_id:
            actual_path = resolve_image_path(img_dir, filename)
            if actual_path is not None:
                actual_filename = os.path.basename(actual_path)
                if actual_filename in filename_to_id:
                    image_id = filename_to_id[actual_filename]
                else:
                    print(f"WARNING: Image {filename} found but not in images list")
                    skipped_rows += 1
                    continue
            else:
                print(f"WARNING: Image {filename} not found in filesystem")
                skipped_rows += 1
                continue
        else:
            image_id = filename_to_id[filename]
        
        img_info = next(img for img in coco_output['images'] if img['id'] == image_id)
        height, width = img_info['height'], img_info['width']

        try:
            xtl = int(row['xtl'])
            ytl = int(row['ytl'])
            xbr = int(row['xbr'])
            ybr = int(row['ybr'])
        except (ValueError, KeyError):
            skipped_rows += 1
            continue

        bbox_width = max(0, xbr - xtl)
        bbox_height = max(0, ybr - ytl)
        bbox = [xtl, ytl, bbox_width, bbox_height]
        area = bbox_width * bbox_height

        cat_id, cat_name = parse_class_field(str(row['class']))

        annotation = {
            "id": current_ann_id,
            "image_id": image_id,
            "category_id": cat_id,
            "category_name": cat_name,
            "segmentation": [],
            "area": area,
            "bbox": bbox,
            "iscrowd": 0,
            # Additional metadata not in the original COCO format
            "detectionID": row['detectionID'],
            "flash": int(row['flash']) if pd.notna(row['flash']) else 0,
            "flower_species": row['flower_species'] if pd.notna(row['flower_species']) else None,
            "focus": row['focus'] if pd.notna(row['focus']) else None,
            "scalingFactor": float(row['scalingFactor']) if pd.notna(row['scalingFactor']) else 1.0,
            "deploymentID": row['deploymentID'],
            "sceneID": row['sceneID'],
            "mediaID": row['mediaID'],
            "eventID": row['eventID'],
            "DOY": int(row['DOY']) if pd.notna(row['DOY']) else None,
            "Hour": int(row['Hour']) if pd.notna(row['Hour']) else None,
            "Minute": int(row['Minute']) if pd.notna(row['Minute']) else None,
            "lifeStage": row['lifeStage'],
            "behaviour": row['behaviour']
        }

        coco_output["annotations"].append(annotation)
        current_ann_id += 1

    with open(OUTPUT_JSON, 'w') as f:
        json.dump(coco_output, f, indent=4)
    
    print(f"Total images in dataset: {len(coco_output['images'])}")
    print(f"Images with annotations: {len(set(ann['image_id'] for ann in coco_output['annotations']))}")
    print(f"Total annotations: {len(coco_output['annotations'])}")
    print(f"Skipped annotation rows: {skipped_rows}")
    print(f"Saved to: {OUTPUT_JSON}")

create_coco(
    csv_path=CSV_FILE, 
    img_dir=IMAGE_DIR
)

In [ ]:
import random
from collections import defaultdict
import matplotlib.pyplot as plt

def plot_annotations_with_crops(coco_json, img_dir, num_images=5, max_crops_per_img=10):
    with open(coco_json) as f:
        coco_data = json.load(f)

    
    images = random.sample(coco_data['images'], k=min(num_images, len(coco_data['images'])))

    img_to_anns = defaultdict(list)
    for ann in coco_data['annotations']:
        img_to_anns[ann['image_id']].append(ann)

    print(f"Visualizing {len(images)} images...")

    for img_info in images:
        img_id = img_info['id']
        filename = img_info['file_name']
        
        img_path = os.path.join(img_dir, filename)
        if not os.path.exists(img_path):
            img_path = os.path.join(img_dir, 'images', filename)
        
        full_img_bgr = cv2.imread(img_path)
        if full_img_bgr is None:
            print(f"Error loading {filename}")
            continue
            
        full_img_rgb = cv2.cvtColor(full_img_bgr, cv2.COLOR_BGR2RGB)
        h, w = full_img_rgb.shape[:2]
        
        anns = img_to_anns[img_id][:max_crops_per_img]
        cols = 1 + len(anns)
        
        _, axes = plt.subplots(1, cols, figsize=(15, 10))
        axes = np.atleast_1d(axes)

        main_view = full_img_rgb.copy()
        for ann in anns:
            bbox = ann['bbox']
            
            if bbox is not None:
                x, y, bw, bh = map(int, bbox)
                cv2.rectangle(main_view, (x, y), (x+bw, y+bh), (255, 0, 0), 2)

        axes[0].imshow(main_view)
        axes[0].set_title(f"{filename}\n({len(anns)} objects)")
        axes[0].axis('off')

        for i, ann in enumerate(anns):
            
            x, y, bw, bh = map(int, ann['bbox'])
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(w, x + bw), min(h, y + bh)
            
            if x2 > x1 and y2 > y1:
                crop = full_img_rgb[y1:y2, x1:x2]
                axes[i+1].imshow(crop)
                axes[i+1].set_title(f"ID: {ann.get('id', i)}\nArea: {int(ann['area'])}")
            else:
                axes[i+1].text(0.5, 0.5, "Invalid Box", ha='center')
            axes[i+1].axis('off')

        plt.tight_layout()
        plt.show()

plot_annotations_with_crops(OUTPUT_JSON, IMAGE_DIR, num_images=3)


## Stratifying script by annotation category (used to create a validation set from the RangeX dataset)

In [ ]:
import json
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split

def stratify_by_attribute(
    input_json, 
    main_json="detections_test_80.json", 
    sample_json="detections_val_20.json", 
    main_ratio=0.8, 
    random_state=42,
    stratify_attr="category_id"
):
    with open(input_json, 'r') as f:
        data = json.load(f)

    images = data.get('images', [])
    annotations = data.get('annotations', [])
    categories = data.get('categories', [])

    # 1. Count frequencies of the target attribute
    attr_counts = Counter(str(ann.get(stratify_attr, '__missing__')) for ann in annotations)

    # 2. Map every image to the unique attributes it contains
    img_to_attrs = defaultdict(set)
    for ann in annotations:
        val = str(ann.get(stratify_attr, '__missing__'))
        img_to_attrs[ann['image_id']].add(val)

    # 3. Assign a primary label based on the rarest attribute value
    image_ids = []
    primary_labels = []

    for img in images:
        img_id = img['id'] 
        img_attrs = img_to_attrs.get(img_id, set())
        
        if not img_attrs:
            primary_label = "__empty__"
        else:
            primary_label = min(img_attrs, key=lambda c: attr_counts[c])
            
        image_ids.append(img_id)
        primary_labels.append(primary_label)

    print(f"Total images to process: {len(image_ids)} (Stratifying by '{stratify_attr}')")

    try:
        main_ids, sample_ids = train_test_split(
            image_ids, 
            train_size=main_ratio, 
            stratify=primary_labels, 
            random_state=random_state
        )
    except ValueError as e:
        print(f"\nA '{stratify_attr}' value has too few instances to stratify perfectly. Falling back to random split.")
        main_ids, sample_ids = train_test_split(
            image_ids, 
            train_size=main_ratio, 
            random_state=random_state
        )

    main_ids_set = set(main_ids)
    sample_ids_set = set(sample_ids)

    def filter_dataset_ids(target_ids_set):
        return {
            "info": data.get("info", {}),
            "licenses": data.get("licenses", []),
            "categories": categories,
            "images": [img for img in images if img['id'] in target_ids_set],
            "annotations": [ann for ann in annotations if ann['image_id'] in target_ids_set]
        }

    main_data = filter_dataset_ids(main_ids_set)
    sample_data = filter_dataset_ids(sample_ids_set)

    print(f"Saving {main_json} with {len(main_data['images'])} images")
    with open(main_json, 'w') as f:
        json.dump(main_data, f, indent=4)

    print(f"Saving {sample_json} with {len(sample_data['images'])} images")
    with open(sample_json, 'w') as f:
        json.dump(sample_data, f, indent=4)

# used to stratify the RangeX dataset by scene_id
# stratify_by_attribute("rangex/detections_coco_last.json", stratify_attr="scene_id")

stratify_by_attribute(os.path.join(DATASETS_PATH, "stefan_smartphone/annotations/cropped/processed/ground_truth_coco.json"))

In [ ]:
from utils.eval_utils import plot_dataset_distribution

ANNOTATIONS_FILE = os.path.join(DATASETS_PATH, "stefan_smartphone/annotations/cropped/processed/ground_truth_coco.json")
IMAGES_DIR = os.path.join(DATASETS_PATH, "stefan_smartphone/cropped")

_ = plot_dataset_distribution(ANNOTATIONS_FILE)
_ = plot_dataset_distribution("detections_test_80.json")
_ = plot_dataset_distribution("detections_val_20.json")